In [1]:
import numpy as np, pandas as pd, joblib, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import features as F

torch.set_grad_enabled(False)
MODELS = ["lr_lex", "rf_lex", "lr_tfidf", "bilstm", "cnn"]

class CharCNN(nn.Module):
    def __init__(self, n_vocab, emb=32, filters=128, kernels=(3, 5, 7), hidden=64, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, emb, padding_idx=0)
        self.convs = nn.ModuleList(nn.Conv1d(emb, filters, k, padding=k // 2) for k in kernels)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(filters * len(kernels), hidden),
                                  nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x):
        mask = (x != 0).unsqueeze(1)
        h = self.emb(x).transpose(1, 2)
        pooled = [torch.relu(c(h)).masked_fill(~mask, 0).amax(dim=2) for c in self.convs]
        return self.head(torch.cat(pooled, dim=1)).squeeze(1)

class CharBiLSTM(nn.Module):
    def __init__(self, n_vocab, emb=32, hidden=64, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(2 * hidden, 1))
    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(x), lengths,
                                                   batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.head(torch.cat([h[0], h[1]], dim=1)).squeeze(1)

def load_net(cls, path):
    sd = torch.load(path, map_location="cpu")
    net = cls(sd["emb.weight"].shape[0]); net.load_state_dict(sd)
    return net.eval()

def predict_net(net, X, bs=2048):
    X = np.where(X < net.emb.num_embeddings, X, F.UNK)
    return np.concatenate([torch.sigmoid(net(torch.from_numpy(X[i:i + bs]).long())).numpy()
                           for i in range(0, len(X), bs)])

vocab, MAX_LEN, mode = F.load_vocab("../data/features/host/vocab.json")
assert mode == "host"
LEX_COLS = pd.read_csv("../data/features/host/lex_train.csv", nrows=1).columns
base = {n: joblib.load(f"../models/host_{n}.joblib") for n in ["lr_lex", "rf_lex", "lr_tfidf"]}
nets = {"cnn": load_net(CharCNN, "../models/host_cnn.pt"),
        "bilstm": load_net(CharBiLSTM, "../models/host_bilstm.pt")}

def score_all(urls):
    urls = list(urls)
    lex = F.lexical_frame(urls, "host").reindex(columns=LEX_COLS, fill_value=0)
    out = {n: base[n].predict_proba(lex)[:, 1] for n in ["lr_lex", "rf_lex"]}
    out["lr_tfidf"] = base["lr_tfidf"].predict_proba([F.to_text(u, "host") for u in urls])[:, 1]
    X = F.encode(urls, vocab, MAX_LEN, "host")
    for n, net in nets.items():
        out[n] = predict_net(net, X)
    return pd.DataFrame(out)[MODELS]

# Operating threshold per model: the score that gave 1% FPR on validation
y_va = np.load("../data/features/host/y_val.npy")
THR = {n: np.quantile(np.load(f"../results/val_probs_host_{n}.npy")[y_va == 0], 0.99) for n in MODELS}

def evaluate(y, P):
    rows = []
    for n in MODELS:
        p = P[n].values; flag = p > THR[n]; half = p >= 0.5
        rows.append({"model": n, "roc_auc": roc_auc_score(y, p),
                     "pr_auc": average_precision_score(y, p),
                     "f1@0.5": f1_score(y, half.astype(int)), "fpr@0.5": half[y == 0].mean(),
                     "recall@thr": flag[y == 1].mean(), "fpr@thr": flag[y == 0].mean()})
    return pd.DataFrame(rows).set_index("model")

print({n: round(float(t), 4) for n, t in THR.items()})

{'lr_lex': 0.7668, 'rf_lex': 0.7833, 'lr_tfidf': 0.7093, 'bilstm': 0.6714, 'cnn': 0.7116}


In [3]:
test = pd.read_csv("../data/splits/test.csv")
test, conf_te = F.to_host_dataset(test)
y_te = test["label"].values
P_te = score_all(test["URL"])
test.reset_index(drop=True).join(P_te).to_csv("../results/phase7_test_scores.csv", index=False)

print(f"test: {len(test):,} hosts, phishing share {y_te.mean():.3f}\n")
print(evaluate(y_te, P_te).round(4).to_string())

# Is the CNN's edge over TF-IDF real? Paired bootstrap on the AUC difference
rng = np.random.default_rng(42)
pc, pt = P_te["cnn"].values, P_te["lr_tfidf"].values
d = []
for _ in range(500):
    i = rng.integers(0, len(y_te), len(y_te))
    d.append(roc_auc_score(y_te[i], pc[i]) - roc_auc_score(y_te[i], pt[i]))
print(f"\nCNN - TF-IDF ROC-AUC: {np.mean(d):+.4f}  95% CI [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]")

test: 33,422 hosts, phishing share 0.395

          roc_auc  pr_auc  f1@0.5  fpr@0.5  recall@thr  fpr@thr
model                                                          
lr_lex     0.8640  0.8704  0.7805   0.0422      0.5953   0.0107
rf_lex     0.8667  0.8732  0.7859   0.0354      0.6387   0.0113
lr_tfidf   0.9145  0.9113  0.8142   0.0419      0.6423   0.0111
bilstm     0.9103  0.9086  0.8003   0.0254      0.6555   0.0119
cnn        0.9202  0.9186  0.8233   0.0268      0.6960   0.0115

CNN - TF-IDF ROC-AUC: +0.0057  95% CI [+0.0035, +0.0078]


In [4]:
zd = pd.read_csv("../data/zeroday/zeroday_eval.csv")
zd = zd.reset_index(drop=True).join(score_all(zd["host"]))
zd.to_csv("../results/phase7_zeroday_scores.csv", index=False)

zd["owner_key"] = zd["owner"].fillna(zd["host"])
slices = {
    "all":           zd,
    "own domain":    zd[zd["slice"] == "own"],
    "platform":      zd[zd["slice"] == "platform"],
    "one per owner": zd.sample(frac=1, random_state=42).drop_duplicates("owner_key"),
}
show = ["roc_auc", "recall@thr", "fpr@thr"]
for name, g in slices.items():
    print(f"\n=== {name}: {int(g.label.sum())} phishing / {int((g.label == 0).sum())} legit ===")
    print(evaluate(g["label"].values, g[MODELS])[show].round(3).to_string())

rows = []
for s in ["pages.dev", "github.io", "vercel.app", "blogspot.com", "workers.dev"]:
    g = zd[zd["platform_suffix"] == s]
    for n in ["lr_tfidf", "cnn"]:
        flag = g[n] > THR[n]
        rows.append({"platform": s, "model": n, "phish": int(g.label.sum()),
                     "legit": int((g.label == 0).sum()),
                     "recall": flag[g.label == 1].mean(), "fpr": flag[g.label == 0].mean()})
print("\n=== per platform, at the val 1%-FPR threshold ===")
print(pd.DataFrame(rows).round(3).to_string(index=False))


=== all: 256 phishing / 3072 legit ===
          roc_auc  recall@thr  fpr@thr
model                                 
lr_lex      0.796       0.758    0.296
rf_lex      0.703       0.738    0.365
lr_tfidf    0.770       0.840    0.430
bilstm      0.804       0.848    0.403
cnn         0.826       0.863    0.445

=== own domain: 157 phishing / 1999 legit ===
          roc_auc  recall@thr  fpr@thr
model                                 
lr_lex      0.866       0.707    0.080
rf_lex      0.834       0.592    0.073
lr_tfidf    0.882       0.745    0.138
bilstm      0.903       0.758    0.127
cnn         0.898       0.783    0.152

=== platform: 99 phishing / 1073 legit ===
          roc_auc  recall@thr  fpr@thr
model                                 
lr_lex      0.701       0.838    0.699
rf_lex      0.587       0.970    0.909
lr_tfidf    0.622       0.990    0.975
bilstm      0.673       0.990    0.919
cnn         0.744       0.990    0.991

=== one per owner: 178 phishing / 3072 legit ===


In [5]:
own = zd[zd["slice"] == "own"]
pd.set_option("display.width", 200)

print("=== Legit own-domain hosts the CNN scores highest (false alarms) ===")
print(own[own.label == 0].nlargest(20, "cnn")[["host", "cnn", "lr_tfidf"]].round(3).to_string(index=False))

print("\n=== Phishing own-domain hosts the CNN scores lowest (misses) ===")
print(own[own.label == 1].nsmallest(15, "cnn")[["host", "cnn", "lr_tfidf"]].round(3).to_string(index=False))

# Feature drift: PhiUSIIL train vs Tranco long-tail legit
lex_tr = pd.read_csv("../data/features/host/lex_train.csv")
y_tr   = np.load("../data/features/host/y_train.npy")
lex_zd = F.lexical_frame(list(own.loc[own.label == 0, "host"]), "host").reindex(columns=LEX_COLS, fill_value=0)
print("\n=== Mean feature values ===")
print(pd.DataFrame({"PhiUSIIL legit": lex_tr[y_tr == 0].mean(),
                    "Tranco legit":   lex_zd.mean(),
                    "PhiUSIIL phish": lex_tr[y_tr == 1].mean()}).round(3).to_string())

print("\n=== CNN score quantiles on legitimate hosts ===")
qs = [0.5, 0.9, 0.99]
print("PhiUSIIL test legit:", np.quantile(P_te["cnn"].values[y_te == 0], qs).round(3))
print("Tranco legit:       ", np.quantile(own.loc[own.label == 0, "cnn"], qs).round(3))

=== Legit own-domain hosts the CNN scores highest (false alarms) ===
                            host  cnn  lr_tfidf
               1xlite-696158.top  1.0     0.994
                    93jk0rvj.top  1.0     0.997
xn--80aweql3c59aea5q.xn--80ao21a  1.0     0.980
 xn--52-6kcanlw5ddbimco.xn--p1ai  1.0     0.991
    xn----7sbkofbbj4akz.xn--90ae  1.0     0.991
                     4626949.xyz  1.0     0.983
                  1xbet-nqaq.top  1.0     0.995
         domlepen-b8v1q7m3ls.xyz  1.0     0.974
   xn--80abnhgmtpfdlnhf.xn--p1ai  1.0     0.995
                 rbt-amp-404.top  1.0     0.994
     xn--31-mlcxldg2d5b.xn--p1ai  1.0     0.996
              xn--q3cpm7dq6h.com  1.0     0.983
                  brad-cdn77.top  1.0     0.995
                  1xbet-ro2b.top  1.0     0.982
            juantoto-805885.shop  1.0     0.967
                  1xbet-auws.top  1.0     0.985
                      007666.com  1.0     0.979
                    site2026.top  1.0     0.992
               clou

In [6]:
import re
own = zd[zd["slice"] == "own"]

def tag(h):
    if "xn--" in h:                                                   return "IDN (non-Latin)"
    if re.search(r"1xbet|casino|slot|toto|poker|judi|gacor|betting", h): return "gambling-style"
    if h.rsplit(".", 1)[-1] in F.ABUSED_TLDS:                          return "abused TLD"
    if sum(c.isdigit() for c in h) >= 3:                                return "digit-heavy"
    return "ordinary"

lo = own[own.label == 0].copy()
lo["tag"]  = lo["host"].map(tag)
lo["flag"] = lo["cnn"] > THR["cnn"]
t = lo.groupby("tag").agg(n=("flag", "size"), flagged=("flag", "sum"))
t["fpr_in_group"] = t["flagged"] / t["n"]
t["share_of_fps"] = t["flagged"] / t["flagged"].sum()
print(t.sort_values("flagged", ascending=False).round(3).to_string())

print("\n=== 20 random 'ordinary' false alarms ===")
fa = lo[lo.flag & (lo.tag == "ordinary")]
print(fa.sample(min(20, len(fa)), random_state=0)[["host", "cnn"]].round(3).to_string(index=False))

                    n  flagged  fpr_in_group  share_of_fps
tag                                                       
ordinary         1804      158         0.088         0.521
abused TLD         74       68         0.919         0.224
digit-heavy        70       47         0.671         0.155
gambling-style     43       22         0.512         0.073
IDN (non-Latin)     8        8         1.000         0.026

=== 20 random 'ordinary' false alarms ===
              host   cnn
    extra-wiki.win 0.892
      xmserver.net 0.835
  compralo.website 0.972
        pixhost.to 0.846
          amcdn.vn 0.907
  webspace-data.io 0.769
  awsdns-cn-57.net 0.997
          sun07.cn 0.809
      inlandfxp.in 0.903
          p8y5.com 0.725
        u7q2mn.com 0.903
     miamiclub.vip 0.992
    ngoaingu24h.vn 0.830
        sp6s8j.com 0.997
         xxgzz.com 0.887
    djflashvpn.lol 0.996
    adstudio.cloud 0.797
         d77.today 0.945
hostingnetweb3.com 0.947
        xiaolu.pro 0.880


In [7]:
import io, zipfile, urllib.request, tldextract

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (student project)"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return r.read()

z = zipfile.ZipFile(io.BytesIO(fetch("https://tranco-list.eu/top-1m.csv.zip")))
tr = pd.read_csv(z.open(z.namelist()[0]), header=None, names=["rank", "domain"])
tr = tr[(tr["rank"] > 10_000) & ~tr["domain"].isin(set(zd["host"]))].sample(5000, random_state=7)

seen = pd.concat([pd.read_csv(f"../data/splits/{s}.csv") for s in ["train", "val", "test"]])
reg = lambda u: tldextract.extract(u).top_domain_under_public_suffix
seen_reg = set(seen["URL"].map(reg))
cal = tr[~tr["domain"].map(reg).isin(seen_reg)].head(2000)
print(f"calibration set: {len(cal)} Tranco domains")

P_cal = score_all(cal["domain"])
THR_CAL = {n: np.quantile(P_cal[n].values, 0.99) for n in MODELS}

rows = []
yo = own["label"].values
for n in MODELS:
    for src, thr in [("PhiUSIIL val", THR[n]), ("Tranco benign", THR_CAL[n])]:
        o, te = own[n].values > thr, P_te[n].values > thr
        rows.append({"model": n, "threshold from": src, "thr": thr,
                     "zeroday recall": o[yo == 1].mean(), "zeroday fpr": o[yo == 0].mean(),
                     "test recall": te[y_te == 1].mean(), "test fpr": te[y_te == 0].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

calibration set: 2000 Tranco domains
   model threshold from   thr  zeroday recall  zeroday fpr  test recall  test fpr
  lr_lex   PhiUSIIL val 0.767           0.707        0.080        0.595     0.011
  lr_lex  Tranco benign 0.996           0.134        0.014        0.258     0.000
  rf_lex   PhiUSIIL val 0.783           0.592        0.073        0.639     0.011
  rf_lex  Tranco benign 0.999           0.102        0.017        0.403     0.001
lr_tfidf   PhiUSIIL val 0.709           0.745        0.138        0.642     0.011
lr_tfidf  Tranco benign 0.988           0.153        0.010        0.272     0.000
  bilstm   PhiUSIIL val 0.671           0.758        0.127        0.655     0.012
  bilstm  Tranco benign 0.998           0.592        0.009        0.393     0.000
     cnn   PhiUSIIL val 0.712           0.783        0.152        0.696     0.012
     cnn  Tranco benign 1.000           0.408        0.012        0.333     0.000


In [8]:
def logits_net(net, urls, bs=2048):
    X = F.encode(list(urls), vocab, MAX_LEN, "host")
    X = np.where(X < net.emb.num_embeddings, X, F.UNK)
    return np.concatenate([net(torch.from_numpy(X[i:i + bs]).long()).double().numpy()
                           for i in range(0, len(X), bs)])

yo = own["label"].values
rows = []
for n, net in nets.items():
    print(f"{n}: hosts at exactly 1.0 -> zero-day own {(own[n] == 1.0).sum()}, "
          f"calibration {(P_cal[n] == 1.0).sum()}")
    z_own, z_cal, z_te = logits_net(net, own["host"]), logits_net(net, cal["domain"]), logits_net(net, test["URL"])
    thr = np.quantile(z_cal, 0.99)
    o, te = z_own > thr, z_te > thr
    rows.append({"model": n, "logit thr": thr,
                 "zeroday auc": roc_auc_score(yo, z_own),
                 "zeroday recall": o[yo == 1].mean(), "zeroday fpr": o[yo == 0].mean(),
                 "test recall": te[y_te == 1].mean(), "test fpr": te[y_te == 0].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

cnn: hosts at exactly 1.0 -> zero-day own 3, calibration 0
bilstm: hosts at exactly 1.0 -> zero-day own 0, calibration 0
 model  logit thr  zeroday auc  zeroday recall  zeroday fpr  test recall  test fpr
   cnn      8.920        0.898           0.408        0.012        0.333       0.0
bilstm      6.045        0.903           0.592        0.009        0.393       0.0


In [9]:
import json
z_cal = {n: logits_net(net, cal["domain"]) for n, net in nets.items()}
thresholds = {n: {"val_prob_1pct": float(THR[n]),
                  "tranco_logit_1pct": float(np.quantile(z_cal[n], 0.99))}
              for n in nets}
with open("../models/thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=1)
print(thresholds)

{'cnn': {'val_prob_1pct': 0.7116237878799438, 'tranco_logit_1pct': 8.920057315826414}, 'bilstm': {'val_prob_1pct': 0.6714159846305847, 'tranco_logit_1pct': 6.044576139450073}}
